In [ ]:
import pandas as pd
import numpy as np

from statsmodels.tsa.statespace.sarimax import SARIMAX
train = pd.read_csv("data/train.csv", parse_dates=["Date"])
store = pd.read_csv("data/store.csv")

df = train.merge(store, on="Store", how="left")

df["DayOfWeek"] = df["Date"].dt.dayofweek + 1
df["Month"] = df["Date"].dt.month

df_store = df[df["Store"] == 1].copy()
df_store = df_store.sort_values("Date")

df_store = df_store[["Date", "Sales", "Promo"]]
df_store = df_store.reset_index(drop=True)

In [2]:
split_index = int(len(df_store) * 0.8)

train = df_store.iloc[:split_index]
test = df_store.iloc[split_index:]

In [3]:
y_train = train["Sales"]
X_train = train[["Promo"]]

y_test = test["Sales"]
X_test = test[["Promo"]]

In [4]:
model = SARIMAX(
    y_train,
    exog=X_train,
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
results = model.fit()
y_pred = results.predict(
    start=len(y_train),
    end=len(y_train) + len(y_test) - 1,
    exog=X_test
)
mae = np.mean(np.abs(y_test - y_pred))
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

nonzero = y_test != 0
mape = np.mean(np.abs((y_test[nonzero] - y_pred[nonzero]) / y_test[nonzero])) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

MAE: 543.7058176759585
RMSE: 982.3664971736789
MAPE: 10.821349088921494
